# LAD-GENERIC validation

Run selected benchmark tasks against adapters stored in Google Drive.
For `open_ended`, the runner evaluates 30 fixed questions and records per-answer perplexity plus unigram, bigram, and trigram repetition.

In [ ]:
%cd /content
!if [ -d lad-generic/.git ]; then git -C lad-generic pull; else git clone https://github.com/RuurdKuiper/lad-generic.git; fi
%cd /content/lad-generic
!python -m pip install --upgrade pip
!python -m pip install --no-cache-dir '.[cuda]'
!python -m pip install --upgrade bitsandbytes
!python -m pip uninstall -y torchao
!nvidia-smi

In [ ]:
import json
import os
import subprocess
from pathlib import Path

from google.colab import drive, userdata

os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/lad-generic-results')
OUTPUTS_DIR = DRIVE_ROOT / 'outputs'
RESULTS_PATH = DRIVE_ROOT / 'validation' / 'benchmark_results.jsonl'

# Adapter directories relative to OUTPUTS_DIR. Examples: 'run-name/best', 'run-name/checkpoint-10000'.
RUNS = [
    'llama-3.1-8b-mask-corrupted-loss/best',
    'llama-3.1-8b-mask-legacysettings-corrupted-loss/best',
]

# Choose any combination of standard tasks and/or 'open_ended'.
TASKS = ['open_ended']
LIMIT = 30  # open_ended has 30 fixed prompts; use a smaller number for a smoke test.
DEVICE = 'cuda'
QUANTIZATION = 'auto'  # 'auto', '4bit', or 'none'
INCLUDE_AUTOREGRESSIVE = False

GENERATION = {
    'max_new_tokens': 256,
    'num_steps': 256,
    'noise_level': 1.0,
    'temperature': 0.7,
    'top_k': 20,
    'seed': 1234,
    'proportional_unmask': True,
}
TASK_GENERATION = {
    'open_ended': {'max_new_tokens': 256, 'num_steps': 256},
}

print('Available adapters:')
from diffusion_lm.inference import find_adapters
print('\n'.join(find_adapters(OUTPUTS_DIR)))

In [ ]:
import yaml

config = {
    'outputs_dir': str(OUTPUTS_DIR),
    'models': RUNS,
    'tasks': TASKS,
    'split': 'test',
    'limit': LIMIT,
    'device': DEVICE,
    'quantization': QUANTIZATION,
    'cache_dir': '/content/huggingface-datasets',
    'results_path': str(RESULTS_PATH),
    'include_autoregressive': INCLUDE_AUTOREGRESSIVE,
    'generation': GENERATION,
    'task_generation': TASK_GENERATION,
}
config_path = Path('/content/benchmark_colab.yaml')
config_path.write_text(yaml.safe_dump(config, sort_keys=False))
print(config_path.read_text())
subprocess.run(['python', 'evaluate_benchmarks.py', '--config', str(config_path)], check=True)

In [ ]:
import pandas as pd

summary_path = RESULTS_PATH.parent / 'benchmark_results_summary.json'
summary = json.loads(summary_path.read_text())
display(pd.DataFrame(summary))

if RESULTS_PATH.exists():
    records = [json.loads(line) for line in RESULTS_PATH.read_text().splitlines() if line.strip()]
    display(pd.DataFrame(records).head())